# choropleth functions

In [ ]:
import plotly.express as px

# Plotly's USA-states location mode needs two-letter postal abbreviations.
state_abbreviations = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR",
    "California": "CA", "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE",
    "District of Columbia": "DC", "Florida": "FL", "Georgia": "GA", "Hawaii": "HI",
    "Idaho": "ID", "Illinois": "IL", "Indiana": "IN", "Iowa": "IA",
    "Kansas": "KS", "Kentucky": "KY", "Louisiana": "LA", "Maine": "ME",
    "Maryland": "MD", "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN",
    "Mississippi": "MS", "Missouri": "MO", "Montana": "MT", "Nebraska": "NE",
    "Nevada": "NV", "New Hampshire": "NH", "New Jersey": "NJ", "New Mexico": "NM",
    "New York": "NY", "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH",
    "Oklahoma": "OK", "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI",
    "South Carolina": "SC", "South Dakota": "SD", "Tennessee": "TN", "Texas": "TX",
    "Utah": "UT", "Vermont": "VT", "Virginia": "VA", "Washington": "WA",
    "West Virginia": "WV", "Wisconsin": "WI", "Wyoming": "WY",
}

excluded_districts = {"ME-1", "ME-2", "NE-1", "NE-2", "NE-3"}

def get_color_maps(ogwin, oglos):

    dem_continuous = [
                    (0, "#b2182b"),       # R: red
                    (0.5, "#f7f7f7"),
                    (1, "#2166ac"),       # D: blue
                ]

    rep_continuous = [
                    (0, "#2166ac"),       
                    (0.5, "#f7f7f7"),
                    (1, "#b2182b"),       
                ]

    dem_discrete = {
                        ogwin: "#2166ac",
                        oglos: "#b2182b",
                        "Tie": "#f7f7f7",
                    }

    rep_discrete = {
                        ogwin: "#b2182b", 
                        oglos: "#2166ac",
                        "Tie": "#f7f7f7",
                    }

    if ogwin.find("(D)") != -1:
        return dem_continuous, dem_discrete
    else:
        return rep_continuous, rep_discrete


def make_election_choropleth(results, title, ogwin, oglos, type):

    # Keep Maine and Nebraska's statewide totals; exclude only their district rows.
    # map results
    state_results = results.loc[~results["States"].isin(excluded_districts)]
    map_df = pd.DataFrame({
        "State": state_results["States"].astype(str).to_numpy(),
        "Margin": pd.to_numeric(state_results[("Margin", "#")]).astype(float).to_numpy(),
        "Margin %": pd.to_numeric(state_results[("Margin", "%")]).astype(float).to_numpy(),
        f"{ogwin} votes": pd.to_numeric(state_results[(ogwin, "#")]).astype(float).to_numpy(),
        f"{oglos} votes": pd.to_numeric(state_results[(oglos, "#")]).astype(float).to_numpy(),
    })
    map_df["Abbreviation"] = map_df["State"].map(state_abbreviations)
    map_df["Winner"] = map_df["Margin"].map(
        lambda margin: ogwin if margin > 0 else (oglos if margin < 0 else "Tie")
    )

    # get color maps
    color_continuous, color_discrete = get_color_maps(ogwin, oglos)

    districts = results.loc[results["States"].isin(excluded_districts)].sort_values(by="States")
    district_notes = []
    for index, row in districts.iterrows():
        margin = int(row[("Margin", "#")])
        winner = ogwin if margin > 0 else (oglos if margin < 0 else "Tie")
        district_notes.append(
            f"{row['States'].to_string(index=False)}: {winner} ({margin:+,} votes)"
        )

    ogwin_ev = int(pd.to_numeric(results[(ogwin, "EV")], errors="coerce").fillna(0).sum())
    oglos_ev = int(pd.to_numeric(results[(oglos, "EV")], errors="coerce").fillna(0).sum())
    ev_note = f"<b>National EV:</b><br>{ogwin} {ogwin_ev} | {oglos} {oglos_ev}"
    
    if type == "Winner":
        fig = px.choropleth(
                map_df,
                locations="Abbreviation",
                locationmode="USA-states",
                color=type,
                category_orders={"Winner": [ogwin, oglos, "Tie"]},
                color_discrete_map=color_discrete,
                hover_name="State",
                hover_data={
                    "Abbreviation": False,
                    "Margin": ":,.0f",
                    "Margin %": ":.2f",
                    "Winner": True,
                    f"{ogwin} votes": ":,.0f",
                    f"{oglos} votes": ":,.0f",
                },
                labels={"Margin": "Margin (votes)"},
                title=title,
            )
    else:
        max_margin = max(abs(map_df[type].min()), abs(map_df[type].max()))
        fig = px.choropleth(
            map_df,
            locations="Abbreviation",
            locationmode="USA-states",
            color=type,
            color_continuous_scale=color_continuous,
            range_color=(-max_margin, max_margin),
            hover_name="State",
            hover_data={
                "Abbreviation": False,
                "Margin": ":,.0f",
                "Margin %": ":.2f",
                "Winner": True,
                f"{ogwin} votes": ":,.0f",
                f"{oglos} votes": ":,.0f",
            },
            labels={"Margin": "Margin (votes)"},
            title=title,
        )
    fig.update_geos(scope="usa", showlakes=True, lakecolor="white")
    fig.update_layout(
        template="plotly_white",
        margin={"l": 100, "r": 0, "t": 55, "b": 10},
        coloraxis_colorbar={"title": "Margin<br>%"},
    )
    fig.add_annotation(
        x=0.10,
        y=0.5,
        xref="paper",
        yref="paper",
        text=ev_note + "<br><br><b>District results (not mapped):</b><br>" + "<br>".join(district_notes),
        showarrow=False,
        align="right",
        xanchor="right",
        yanchor="middle",
        font={"size": 7},
    )
    return fig




In [10]:
def test_function():
    print('test 2d')